# GAN with Keras on Fashion MNIST

This notebook implements a Generative Adversarial Network (GAN) using TensorFlow/Keras to generate Fashion MNIST images.

Based on: https://colab.research.google.com/github/qaazii/3D-Bounding-Boxes-From-Monocular-Images/blob/master/Intro-to-Generative-Adversarial-Network/Tensorflow/GAN_Tensorflow_Fashion_MNIST.ipynb

In [ ]:
# Install required packages (uncomment if running in Google Colab)
# !pip install numpy
# !pip install tensorflow
# !pip install matplotlib

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from IPython import display
import matplotlib.pyplot as plt
%matplotlib inline
from tensorflow import keras

In [ ]:
print('TensorFlow version:', tf.__version__)

## Load and Preprocess Data

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1).astype('float32')
x_train = (x_train - 127.5) / 127.5  # Normalize the images to [-1, 1]

# Batch and shuffle the data
BUFFER_SIZE = 60000
BATCH_SIZE = 128

train_dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

print('Training data shape:', x_train.shape)

## Hyperparameters

In [ ]:
latent_dim = 100
image_dim = 784  # 28 * 28
num_examples_to_generate = 25
learning_rate = 0.0002

# We will reuse this seed to visualize progress over training
seed = tf.random.normal([num_examples_to_generate, latent_dim])

## Generator Model

The generator takes a random noise vector (latent space) as input and produces a 28x28x1 image.

In [ ]:
def build_generator(image_dim):
    inputs = keras.Input(shape=(100,), name='input_layer')
    x = layers.Dense(128, kernel_initializer=tf.keras.initializers.he_uniform(), name='dense_1')(inputs)
    x = layers.LeakyReLU(0.2, name='leaky_relu_1')(x)
    x = layers.Dense(256, kernel_initializer=tf.keras.initializers.he_uniform(), name='dense_2')(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=0.8, name='bn_1')(x)
    x = layers.LeakyReLU(0.2, name='leaky_relu_2')(x)
    x = layers.Dense(512, kernel_initializer=tf.keras.initializers.he_uniform(), name='dense_3')(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=0.8, name='bn_2')(x)
    x = layers.LeakyReLU(0.2, name='leaky_relu_3')(x)
    x = layers.Dense(1024, kernel_initializer=tf.keras.initializers.he_uniform(), name='dense_4')(x)
    x = layers.BatchNormalization(momentum=0.1, epsilon=0.8, name='bn_3')(x)
    x = layers.LeakyReLU(0.2, name='leaky_relu_4')(x)
    x = layers.Dense(image_dim, kernel_initializer=tf.keras.initializers.he_uniform(), activation='tanh', name='dense_5')(x)
    outputs = layers.Reshape((28, 28, 1), name='Reshape_Layer')(x)
    model = tf.keras.Model(inputs, outputs, name='Generator')
    return model

generator = build_generator(image_dim)
generator.summary()

## Discriminator Model

The discriminator takes a 28x28x1 image as input and outputs a probability indicating whether the image is real or fake.

In [ ]:
def build_discriminator():
    inputs = keras.Input(shape=(28, 28, 1), name='input_layer')
    x = layers.Flatten(name='flatten_layer')(inputs)
    x = layers.Dense(512, kernel_initializer=tf.keras.initializers.he_uniform(), name='dense_1')(x)
    x = layers.LeakyReLU(0.2, name='leaky_relu_1')(x)
    x = layers.Dense(256, kernel_initializer=tf.keras.initializers.he_uniform(), name='dense_2')(x)
    x = layers.LeakyReLU(0.2, name='leaky_relu_2')(x)
    outputs = layers.Dense(1, kernel_initializer=tf.keras.initializers.he_uniform(), activation='sigmoid', name='dense_3')(x)
    model = tf.keras.Model(inputs, outputs, name='Discriminator')
    return model

discriminator = build_discriminator()
discriminator.summary()

## Loss Functions

In [ ]:
binary_cross_entropy = tf.keras.losses.BinaryCrossentropy()

def generator_loss(fake_output):
    return binary_cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = binary_cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = binary_cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

## Optimizers

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.5, beta_2=0.999)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate, beta_1=0.5, beta_2=0.999)

## Checkpoints

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')
checkpoint = tf.train.Checkpoint(
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator=generator,
    discriminator=discriminator
)

## Training Step

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_gen, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_disc, discriminator.trainable_variables))

## Image Generation and Saving

In [ ]:
os.makedirs('generated_images', exist_ok=True)

def generate_and_save_images(model, epoch, test_input):
    # training=False so all layers run in inference mode (batchnorm)
    predictions = model(test_input, training=False)

    fig = plt.figure(figsize=(5, 5))
    for i in range(predictions.shape[0]):
        plt.subplot(5, 5, i + 1)
        pred = (predictions[i, :, :, 0] + 1) * 127.5
        pred = np.array(pred)
        plt.imshow(pred.astype(np.uint8), cmap='gray')
        plt.axis('off')

    plt.suptitle(f'Epoch {epoch}', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'generated_images/image_at_epoch_{epoch:04d}.png')
    plt.show()

## Training Loop

In [ ]:
def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()

        for image_batch in dataset:
            train_step(image_batch)

        display.clear_output(wait=True)
        generate_and_save_images(generator, epoch + 1, seed)

        # Save checkpoint every 15 epochs
        if (epoch + 1) % 15 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

        print(f'Time for epoch {epoch + 1} is {time.time() - start:.2f} sec')

    # Generate images after the final epoch
    display.clear_output(wait=True)
    generate_and_save_images(generator, epochs, seed)

In [ ]:
EPOCHS = 200
train(train_dataset, EPOCHS)